# Athena 02 — Register Yelp Reviews CSV as an Athena Table

Creates an external Athena table over the raw-reviews CSV that notebook `01_setup_S3_bucket.ipynb` uploaded to `s3://<bucket>/raw/reviews/`.

In [1]:
import boto3
import pandas as pd
from pyathena import connect

%store -r bucket
%store -r region
%store -r database_name
%store -r s3_staging_dir
%store -r raw_reviews_prefix

table_name = "reviews_raw"
s3_reviews_location = f"s3://{bucket}/{raw_reviews_prefix}/"

print("Database:        ", database_name)
print("Table to create: ", table_name)
print("S3 location:     ", s3_reviews_location)

%store table_name
%store s3_reviews_location

conn = connect(region_name=region, s3_staging_dir=s3_staging_dir)

Database:         yelp_db
Table to create:  reviews_raw
S3 location:      s3://yelp-sentiment-mlops-965705611982/raw/reviews/
Stored 'table_name' (str)
Stored 's3_reviews_location' (str)


In [2]:
drop_statement = f"DROP TABLE IF EXISTS {database_name}.{table_name}"
print(drop_statement)
pd.read_sql(drop_statement, conn)

DROP TABLE IF EXISTS yelp_db.reviews_raw


/tmp/ipykernel_83943/2115113509.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(drop_statement, conn)


""


In [3]:
create_statement = f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {database_name}.{table_name} (
    review_id   STRING,
    business_id STRING,
    user_id     STRING,
    stars       DOUBLE,
    review_text STRING,
    date        STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    'separatorChar' = ',',
    'quoteChar'     = '\\"',
    'escapeChar'    = '\\\\'
)
LOCATION '{s3_reviews_location}'
TBLPROPERTIES ('skip.header.line.count'='1')
"""
print(create_statement)
pd.read_sql(create_statement, conn)


CREATE EXTERNAL TABLE IF NOT EXISTS yelp_db.reviews_raw (
    review_id   STRING,
    business_id STRING,
    user_id     STRING,
    stars       DOUBLE,
    review_text STRING,
    date        STRING
)
ROW FORMAT SERDE 'org.apache.hadoop.hive.serde2.OpenCSVSerde'
WITH SERDEPROPERTIES (
    'separatorChar' = ',',
    'quoteChar'     = '\"',
    'escapeChar'    = '\\'
)
LOCATION 's3://yelp-sentiment-mlops-965705611982/raw/reviews/'
TBLPROPERTIES ('skip.header.line.count'='1')



/tmp/ipykernel_83943/2908604259.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  pd.read_sql(create_statement, conn)


""


In [4]:
df_show = pd.read_sql(f"SHOW TABLES IN {database_name}", conn)
df_show

/tmp/ipykernel_83943/3793194707.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_show = pd.read_sql(f"SHOW TABLES IN {database_name}", conn)


,tab_name
0,reviews_raw


## Run a sample query to confirm registration

In [5]:
sample_df = pd.read_sql(
    f"SELECT review_id, stars, SUBSTR(review_text, 1, 80) AS review_snippet FROM {database_name}.{table_name} LIMIT 5",
    conn,
)
sample_df

/tmp/ipykernel_83943/3827926199.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sample_df = pd.read_sql(


,review_id,stars,review_snippet
0,Ax8Ft5uvWWNTM0nsotGeQA,5.0,Most people would argue that St. Louis is a be...
1,KpYV_3KQ-IIhJiPqY9rUbA,5.0,Extremely unique with a delicious 7 course mea...
2,1haGVmQWr3-d4SZ7fMLMqw,4.0,You know they are authentic when they only acc...
3,LKwNWaICsB54vjfeh7nYxg,5.0,Ordered take out last night and the food was a...
4,r-q9R-EHTo0mH4EmQF2Z7A,3.0,I am not a very big fan of Chili's. I don't fi...


In [6]:
count_df = pd.read_sql(
    f"SELECT COUNT(*) AS row_count FROM {database_name}.{table_name}",
    conn,
)
count_df

/tmp/ipykernel_83943/3213435175.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  count_df = pd.read_sql(


,row_count
0,300000


In [7]:
ingest_create_athena_table_passed = table_name in df_show.values
%store ingest_create_athena_table_passed
print("Athena reviews_raw table registered:", ingest_create_athena_table_passed)

Stored 'ingest_create_athena_table_passed' (bool)
Athena reviews_raw table registered: True


## Done

Continue to `03_Convert_CSV_To_Parquet_With_Athena.ipynb` to materialize a Parquet copy of the data for faster downstream queries.